In [1]:
import sys
sys.path.append('../')

# import os
# os.chdir("../")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.datagen import load_s3_data_as_df, load_local_data_as_df
from utils.utils import json_numpy_serialzer
from utils.logging import LOGGER

from sanitisation_techniques.sanitiser import SanitiserNHS
from generative_models.data_synthesiser import (IndependentHistogram,
                                                BayesianNet,
                                                PrivBayes,
                                                Cvine,
                                                CvineSensitive,
                                                IMRV)
from generative_models.CTGAN import CTGAN
from generative_models.TVAE import TVAE
from generative_models.pate_gan import PATEGAN
from predictive_models.predictive_model import RandForestClassTask, LogRegClassTask, LinRegTask

SEED = 42

/uio/hume/student-u82/beritoo/Documents/synthetic_data_release/.venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))
2026-08-19 11:23:15.198887: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-19 11:23:17,539:tensorflow:WARNING:From /uio/hume/student-u82/beritoo/Documents/synthetic_data_release/.venv/lib64/python3.9/site-packages/tensorflow/python/compat/v2_compat.py:98: disable_resource_variables (from tensorflow.python.ops.resource_variables_toggle) is deprecated and will be removed in a future version.
Instructions for updating:
non-resource variables are not supported in the long term


In [3]:
# Read real data (look at a more realistic example!!)
s3name = "../data/simulated_data/real_data"
data, metadata = load_local_data_as_df(s3name)
sensitive = ["x1", "x6", "x12"]

In [4]:
# Train-test-split
train_ind = np.random.choice([True, False], data.shape[0], replace=True, p=[0.7,0.3])
data_train = data.iloc[train_ind]
data_test = data.iloc[~train_ind]

In [5]:
# Initialize random forest classifier           
rf = RandForestClassTask(metadata=metadata, labelCol="y")

In [6]:
# Train on real test on real performance
rf.train(data_train)
rf.get_metrics(data_test)

{'Accuracy': 0.9772727272727273,
 'BalancedAccuracy': 0.9771729957805908,
 'nClassesTrain': 2,
 'F1': 0.9779179810725552,
 'AUC-ROC': 0.9978481012658227,
 'AUC-PR': 0.9977502355551651}

In [7]:
# Train on synthetic test on synthetic performance
gmList = [PrivBayes(metadata, 25, 1, 0.1),
          Cvine(metadata, "all"),
        #   CvineSensitive(metadata, "all", sensitive, 5),
          IMRV(metadata, "all", sensitive, 0.5),
          CTGAN(metadata, 400,100),
          TVAE(metadata, 800, 100)];

In [8]:
acc_results = {}
auc_results = {}
for gm in gmList:
    print(f"Running utility evaluation for {gm.__name__}")
    gm.fit(data_train)
    synData = [gm.generate_samples(500) for _ in range(50)]
    acc = auc = np.zeros(len(synData))
    for i, synDat in enumerate(synData):
        rf.train(synDat)
        metrics = rf.get_metrics(data_test)
        acc[i] = metrics['Accuracy'] 
        auc[i] = metrics["AUC-ROC"]
    acc_results[gm.__name__] = acc
    auc_results[gm.__name__] = auc

Running utility evaluation for PrivBayes(epsilon=0.1)
Running utility evaluation for Cvine(trunc_lvl=20)
Running utility evaluation for IMRV(weight=0.5, trunc_lvl=20)


2026-08-19 11:25:51,284:sdv.metadata.single_table:INFO:Detected metadata:
2026-08-19 11:25:51,284:sdv.metadata.single_table:INFO:{
    "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1",
    "columns": {
        "x1": {
            "sdtype": "numerical"
        },
        "x2": {
            "sdtype": "numerical"
        },
        "x3": {
            "sdtype": "numerical"
        },
        "x4": {
            "sdtype": "numerical"
        },
        "x5": {
            "sdtype": "numerical"
        },
        "x6": {
            "sdtype": "numerical"
        },
        "x7": {
            "sdtype": "numerical"
        },
        "x8": {
            "sdtype": "numerical"
        },
        "x9": {
            "sdtype": "numerical"
        },
        "x10": {
            "sdtype": "numerical"
        },
        "x11": {
            "sdtype": "numerical"
        },
        "x12": {
            "sdtype": "numerical"
        },
        "x13": {
            "sdtype": "numerical"
        },
       

Running utility evaluation for CTGAN(epochs=400, batch_size=100)


2026-08-19 11:25:51,765:rdt.transformers.null:INFO:Guidance: There are no missing values in column x2. Extra column not created.
2026-08-19 11:25:51,853:rdt.transformers.null:INFO:Guidance: There are no missing values in column x3. Extra column not created.
2026-08-19 11:25:51,939:rdt.transformers.null:INFO:Guidance: There are no missing values in column x4. Extra column not created.
2026-08-19 11:25:52,024:rdt.transformers.null:INFO:Guidance: There are no missing values in column x5. Extra column not created.
2026-08-19 11:25:52,110:rdt.transformers.null:INFO:Guidance: There are no missing values in column x6. Extra column not created.
2026-08-19 11:25:52,197:rdt.transformers.null:INFO:Guidance: There are no missing values in column x7. Extra column not created.
2026-08-19 11:25:52,283:rdt.transformers.null:INFO:Guidance: There are no missing values in column x8. Extra column not created.
2026-08-19 11:25:52,367:rdt.transformers.null:INFO:Guidance: There are no missing values in colum

Running utility evaluation for TVAE(epochs=800, batch_size=100)


2026-08-19 11:26:57,365:rdt.transformers.null:INFO:Guidance: There are no missing values in column x2. Extra column not created.
2026-08-19 11:26:57,450:rdt.transformers.null:INFO:Guidance: There are no missing values in column x3. Extra column not created.
2026-08-19 11:26:57,534:rdt.transformers.null:INFO:Guidance: There are no missing values in column x4. Extra column not created.
2026-08-19 11:26:57,617:rdt.transformers.null:INFO:Guidance: There are no missing values in column x5. Extra column not created.
2026-08-19 11:26:57,722:rdt.transformers.null:INFO:Guidance: There are no missing values in column x6. Extra column not created.
2026-08-19 11:26:57,805:rdt.transformers.null:INFO:Guidance: There are no missing values in column x7. Extra column not created.
2026-08-19 11:26:57,888:rdt.transformers.null:INFO:Guidance: There are no missing values in column x8. Extra column not created.
2026-08-19 11:26:57,972:rdt.transformers.null:INFO:Guidance: There are no missing values in colum

In [ ]:
auc_results

In [ ]:
pd.DataFrame(auc_results).boxplot(rot=90, grid=False)

In [9]:
metrics

{'Accuracy': 0.9155844155844156,
 'BalancedAccuracy': 0.9163713080168776,
 'nClassesTrain': 2,
 'F1': 0.9150326797385621,
 'AUC-ROC': 0.9785232067510549,
 'AUC-PR': 0.9811627115577951}